In [1]:
import sys
from pathlib import Path

root_dir = Path.cwd().parent 
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

import pandas as pd
import io
from utils.connections import get_azure_connection, get_postgres_connection

In [2]:
with get_azure_connection() as conn:
    with conn.cursor() as cursor:
        cursor.execute("""
            SELECT 
                a.*,
                'dbo.nowrthwind' as origem
            FROM territories a;
            """)
        rows= cursor.fetchall()
        columns = [column[0] for column in cursor.description]
        df = pd.DataFrame(rows, columns = columns)

In [3]:
df

,TerritoryID,TerritoryDescription,RegionID,origem
0,01581,Westboro ...,1,dbo.nowrthwind
1,01730,Bedford ...,1,dbo.nowrthwind
2,01833,Georgetow ...,1,dbo.nowrthwind
3,02116,Boston ...,1,dbo.nowrthwind
4,02139,Cambridge ...,1,dbo.nowrthwind
5,02184,Braintree ...,1,dbo.nowrthwind
6,02903,Providence ...,1,dbo.nowrthwind
7,03049,Hollis ...,3,dbo.nowrthwind
8,03801,Portsmouth ...,3,dbo.nowrthwind
9,06897,Wilton ...,1,dbo.nowrthwind


In [5]:
with get_postgres_connection() as conn:
    with conn.cursor() as cursor:
        # Limpa a tabela antes da nova carga
        cursor.execute("TRUNCATE TABLE staging.territories;")
        
        # Move os dados para o buffer
        csv_buffer = io.StringIO()
        df.to_csv(csv_buffer, index=False, header=True)
        csv_buffer.seek(0)
        
        # Executa o COPY para a tabela
        copy_sql = """
            COPY staging.territories (
                TerritoryID,
                TerritoryDescription,
                RegionID,
                origem
            ) 
            FROM STDIN WITH CSV HEADER
        """
        cursor.copy_expert(copy_sql, csv_buffer)
        conn.commit()